Import Libraries

In [1]:
import os
import sys

os.makedirs("Shopease/src", exist_ok=True)
# Add the parent directory of 'src' (which is 'Shopease') to sys.path
sys.path.insert(0, os.path.abspath('Shopease'))

In [2]:
%%writefile Shopease/src/preprocessing.py
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def lowercase(text):
    return str(text).lower()

def remove_urls(text):
    return re.sub(r"http\S+|www\S+", "", text)

def remove_email(text):
    return re.sub(r"\S+@\S+", "", text)

def remove_numbers(text):
    return re.sub(r"\d+", "", text)

def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

def remove_spaces(text):
    return re.sub(r"\s+", " ", text).strip()

def tokenize(text):
    return word_tokenize(text)

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

def lemmatize(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

def join_words(tokens):
    return " ".join(tokens)

def clean_text(text):
    text = lowercase(text)
    text = remove_urls(text)
    text = remove_email(text)
    text = remove_numbers(text)
    text = remove_punctuation(text)
    text = remove_spaces(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize(tokens)
    return join_words(tokens)

Overwriting Shopease/src/preprocessing.py


In [3]:
pip install langdetect

In [4]:
pip install src

  Using cached src-0.0.7.zip (6.3 kB)
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for src
  Running setup.py clean for src
Failed to build src
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (src)


In [5]:
import warnings
warnings.filterwarnings("ignore")

import re
import string
import joblib
import pandas as pd
import numpy as np

import nltk
nltk.download('stopwords') # Add this line to download stopwords
nltk.download('punkt_tab') # Add this line to download punkt_tab
nltk.download('wordnet') # Add this line to download wordnet

from langdetect import detect
from langdetect import DetectorFactory

from nltk.corpus import stopwords

from nltk.stem import WordNetLemmatizer
from src.preprocessing import clean_text
from tqdm import tqdm

tqdm.pandas()

DetectorFactory.seed = 42

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Apply Text Preprocessing

In [6]:
import pandas as pd

df = pd.read_csv('/content/eda_completed.csv')
df['cleaned_review'] = df['review'].progress_apply(clean_text)
df.head()

100%|██████████| 21054/21054 [00:14<00:00, 1493.29it/s]


,review_id,product_category,timestamp,country,rating,review,sentiment,date,year,month,month_name,weekday,hour,review_length,cleaned_review
0,REV-50BCBCD9,Sports,2024-09-16 13:44:26+00:00,US,1,"I registered on the website, tried to order a ...",Negative,2024-09-16,2024,9,September,Monday,13,106,registered website tried order laptop entered ...
1,REV-6D2B2651,Toys,2024-09-16 18:26:46+00:00,GB,1,Had multiple orders one turned up and driver h...,Negative,2024-09-16,2024,9,September,Monday,18,53,multiple order one turned driver phone door nu...
2,REV-F7E80372,Toys,2024-09-16 21:47:39+00:00,GB,1,I informed these reprobates that I WOULD NOT B...,Negative,2024-09-16,2024,9,September,Monday,21,122,informed reprobate would going visit sick rela...
3,REV-ED2B173F,Sports,2024-09-17 07:15:49+00:00,AU,1,I have bought from Amazon before and no proble...,Negative,2024-09-17,2024,9,September,Tuesday,7,82,bought amazon problem happy service price amaz...
4,REV-E48A7AB9,Fashion,2024-09-16 18:37:17+00:00,GB,1,If I could give a lower rate I would! I cancel...,Negative,2024-09-16,2024,9,September,Monday,18,100,could give lower rate would cancelled amazon p...


####Language Detection

In [7]:
def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"

df['language'] = df['cleaned_review'].progress_apply(detect_language)
df['language'].value_counts()

100%|██████████| 21054/21054 [01:17<00:00, 273.33it/s]


,count
language,
en,20419
it,117
da,116
fr,62
ro,51
nl,38
af,33
no,32
es,28


####Review Statistics

#####Word Count

In [8]:
df["word_count"] = (

    df["cleaned_review"]

    .str.split()

    .apply(len)

)

####Character Count

In [9]:
df["character_count"] = (

    df["cleaned_review"]

    .str.len()

)

####Sentence Count

In [10]:
df["sentence_count"] = (

    df["review"]

    .apply(

        lambda x:

        len(

            nltk.sent_tokenize(

                str(x)

            )

        )

    )

)

####Average Word Length

In [11]:
df["avg_word_length"] = (

    df["character_count"]

    /

    df["word_count"]

)

####Save Dataset

In [12]:
import os

os.makedirs("Shopease/data/processed", exist_ok=True)

df.to_csv(
    "Shopease/data/processed/processed_reviews.csv",
    index=False
)

####Save Language Statistics

In [13]:
language_summary = (

    df["language"]

    .value_counts()

    .reset_index()

)

language_summary.columns = [

    "Language",

    "Reviews"

]

In [14]:
import os

os.makedirs("Shopease/outputs/reports", exist_ok=True)

language_summary.to_csv(

    "Shopease/outputs/reports/language_summary.csv",

    index=False

)

####Save Feature Summary

In [15]:
summary = pd.DataFrame({

    "Metric":[

        "Average Word Count",

        "Average Character Count",

        "Average Sentence Count"

    ],

    "Value":[

        df["word_count"].mean(),

        df["character_count"].mean(),

        df["sentence_count"].mean()

    ]

})

In [16]:
summary.to_csv(

    "Shopease/outputs/reports/text_summary.csv",

    index=False

)